# PhyDG-Slide: full experiment grid on a GPU (Colab or Kaggle)

1. Runtime -> change runtime type -> GPU (T4 or better). On Kaggle: Settings -> Accelerator -> GPU, Internet on.
2. Run the cells in order. Runs that finished are skipped, so after a disconnect just run everything again.
3. Data (~15 GB of NetCDF after filtering, ~3.7 GB cache) needs ~20 GB of disk. Colab gives ~100 GB; on Kaggle use /kaggle/temp or mount persistent storage.

With `USE_DRIVE = True` (default), the cache and results are kept in `MyDrive/phydg-slide`, about 3.5 GB of Drive space.

In [ ]:
import os
WORK = "/content" if os.path.exists("/content") else "/kaggle/working"
REPO = f"{WORK}/abhishekjadhavcs-afk-repo"
BRANCH = "claude/phydg-slide"
%cd {WORK}
!test -d {REPO} || git clone -b {BRANCH} https://github.com/abhishekjadhavcs-afk/abhishekjadhavcs-afk-repo {REPO}
%cd {REPO}
!git pull -q
!pip install -q xarray h5netcdf netCDF4
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# Optional (Colab): keep the composite cache (~3.5 GB) and results in Google Drive so a
# disconnect does not cost the cache build or finished runs. Raw NetCDF stays on local disk.
USE_DRIVE = True
if USE_DRIVE and os.path.exists("/content"):
    from google.colab import drive
    drive.mount("/content/drive")
    PERSIST = "/content/drive/MyDrive/phydg-slide"
    !mkdir -p {PERSIST}/cache {PERSIST}/runs data
    !test -L data/cache || ln -s {PERSIST}/cache data/cache
    !test -L runs || ln -s {PERSIST}/runs runs
    !ls -la data runs | head

In [ ]:
# Stream the S2 archives (~39 GB downloaded, ~15 GB kept). Resumable.
# Skipped automatically if the cache in Drive is already complete.
import glob
if len(glob.glob("data/cache/*.npz")) < 4988:
    !python scripts/download_s2.py --out data/data_harmonized
else:
    print("cache complete, skipping download")

In [ ]:
# Precompute composites once (~3.7 GB). Resumable.
!python scripts/build_cache.py --data-root data/data_harmonized --out data/cache --workers 2

In [ ]:
!python -m pytest -q tests

## Stage 1: main comparison (U-Net vs PhyDG-Slide, 8 folds x 3 seeds)
About 30 min per run on a T4 (estimate), so ~24 h for 48 runs. Use fewer seeds first if GPU hours are limited.

In [ ]:
!python scripts/run_grid.py --variants unet phydg --seeds 42 123 777 -- --workers 2 --epochs 60

## Stage 2: ablations (single seed)

In [ ]:
!python scripts/run_grid.py --variants phydg-nostyle phydg-noibn phydg-nofilm phydg-nopoe phydg-novrex --seeds 42 -- --workers 2 --epochs 60

In [ ]:
!python scripts/run_grid.py --summary-only
!tar -czf results.tar.gz runs/*/*/results.json && ls -lh results.tar.gz